In [1]:
import torch
import numpy as np
import scipy
from scipy.integrate import quad
import matplotlib.pyplot as plt

In [2]:
def beta(p, omega, A, B, kappa):
    D = 1/kappa * np.log(A) - B

    pref = 1j / 2*np.pi / (p * omega)
    exp = np.exp(-np.pi*p / (2*kappa))
    phase = np.exp(-1j * p * D) * np.exp(-1j * omega * B) * (omega**(-1j * p / kappa))
    gam = scipy.special.gamma(1 + 1j * p / kappa)

    return pref * exp * phase * gam

def alpha(p, omega, A, B, kappa):
    D = 1/kappa * np.log(A) - B

    pref = -1j / 2*np.pi / (p * omega)
    exp = np.exp(np.pi*p / (2*kappa))
    phase = np.exp(-1j * p * D) * np.exp(1j * omega * B) * (omega**(-1j * p / kappa))
    gam = scipy.special.gamma(1 + 1j * p / kappa)

    return pref * exp * phase * gam

In [3]:
A = 1
B = 10
kappa = 20

In [4]:
beta_formula = np.zeros((10, 10), dtype=np.complex64)
alpha_formula = np.zeros((10, 10), dtype=np.complex64)

In [5]:
for i in range(1, 11):
    for j in range(1, 11):
        beta_formula[i-1][j-1] = beta(i, j, A, B, kappa)
        alpha_formula[i-1][j-1] = alpha(i, j, A, B, kappa)

In [6]:
beta_formula

array([[ 4.17458080e-02+1.44856489e+00j, -4.31956112e-01-5.81751347e-01j,
         4.55946118e-01+1.59548402e-01j, -3.50758761e-01+9.06833932e-02j,
         1.93587884e-01-2.15701297e-01j, -3.53968441e-02+2.38919854e-01j,
        -8.74007344e-02-1.87669784e-01j,  1.54141545e-01+9.51534510e-02j,
        -1.60955086e-01+4.51747701e-03j,  1.18900925e-01-8.28456283e-02j],
       [ 3.29583138e-01-5.78443825e-01j,  4.20418940e-02+3.30209106e-01j,
        -1.50029495e-01-1.63517818e-01j,  1.62264422e-01+3.70358601e-02j,
        -1.23987652e-01+4.85381819e-02j,  6.30363226e-02-9.13134888e-02j,
        -1.29042077e-03+9.50983092e-02j, -4.52576801e-02-6.98361769e-02j,
         6.78770542e-02+2.94038374e-02j, -6.55351654e-02+1.17202867e-02j],
       [-3.55339289e-01+1.96713448e-01j,  7.64566883e-02-1.88135386e-01j,
         3.34997214e-02+1.31175086e-01j, -7.75049925e-02-6.55981228e-02j,
         8.08748156e-02+7.59949302e-03j, -5.91153130e-02+3.29797380e-02j,
         2.59444136e-02-5.18985838e-

In [7]:
alpha_formula

array([[ 1.52746284e+00-0.7362725j , -8.44287574e-01-0.07738491j,
         4.51206058e-01+0.34041765j, -1.50764167e-01-0.39619774j,
        -6.75267205e-02+0.3323398j ,  1.96035489e-01-0.20356292j,
        -2.35480845e-01+0.05680785j,  2.00394183e-01+0.06904922j,
        -1.16943806e-01-0.1477191j ,  1.68747827e-02+0.16872355j],
       [-9.07149613e-01-0.08877133j,  3.75248462e-01+0.25862864j,
        -1.27392828e-01-0.27583015j, -2.58728527e-02+0.2263971j ,
         1.12730727e-01-0.14326136j, -1.42853528e-01+0.0516785j ,
         1.27277553e-01+0.02748711j, -8.14325288e-02-0.07968692j,
         2.33634785e-02+0.09854414j,  2.96995491e-02-0.08617392j],
       [ 5.19995868e-01+0.39109096j, -1.42880827e-01-0.29227048j,
        -1.29407579e-02+0.21649754j,  9.07386914e-02-0.1350027j ,
        -1.17886335e-01+0.05510649j,  1.07779406e-01+0.01196924j,
        -7.32758865e-02-0.05718738j,  2.81101856e-02+0.07631925j,
         1.46920383e-02-0.07078602j, -4.50162776e-02+0.04697886j],
       

In [8]:
def f(v, A, B, kappa):
    return (1/kappa) * np.log((B-v) / A) - B

In [9]:
def alpha_int(p, omega, A, B, kappa):
    pref = 1 / (2*np.pi) * np.sqrt(omega/p)

    def integrand_real(x):
        theta = p * f(x, A, B, kappa) + omega * x
        return np.cos(theta)
    
    def integrand_imag(x):
        theta = p * f(x, A, B, kappa) + omega * x
        return -np.sin(theta)

    real_part, _ = quad(integrand_real, 0, B)
    imag_part, _ = quad(integrand_imag, 0, B)

    return pref * (real_part + 1j*imag_part)

def beta_int(p, omega, A, B, kappa):
    pref = -1 / (2*np.pi) * np.sqrt(omega/p)

    def integrand_real(x):
        theta = p * f(x, A, B, kappa) - omega * x
        return np.cos(theta)
    
    def integrand_imag(x):
        theta = p * f(x, A, B, kappa) - omega * x
        return -np.sin(theta)

    real_part, _ = quad(integrand_real, 0, B)
    imag_part, _ = quad(integrand_imag, 0, B)

    return pref * (real_part + 1j*imag_part)

In [10]:
beta_integral = np.zeros((10, 10), dtype=np.complex64)
alpha_integral = np.zeros((10, 10), dtype=np.complex64)

In [11]:
for i in range(1, 11):
    for j in range(1, 11):
        beta_integral[i-1][j-1] = beta_int(i, j, A, B, kappa)
        alpha_integral[i-1][j-1] = alpha_int(i, j, A, B, kappa)

In [12]:
beta_integral

array([[-0.20597701+0.19797361j,  0.05152571+0.12308189j,
        -0.09894405+0.02057473j, -0.02305927+0.1436044j ,
        -0.00485474+0.00372763j, -0.07937535+0.09038803j,
         0.02879928+0.05472111j, -0.06789123+0.02118679j,
        -0.00483163+0.09273437j, -0.01364945-0.00053915j],
       [ 0.18187068-0.04749104j,  0.01881886-0.09895031j,
         0.05693081+0.01558191j,  0.06776632-0.07572664j,
         0.00188363-0.01119535j,  0.07513349-0.02289211j,
         0.00660809-0.04940717j,  0.04047676+0.0084744j ,
         0.03985837-0.05286694j,  0.00452737-0.00294897j],
       [-0.13776067-0.03461593j, -0.05948186+0.06249039j,
        -0.02997983-0.02486636j, -0.07787033+0.02230862j,
        -0.00909378+0.01459432j, -0.05662362-0.01416055j,
        -0.02990097+0.03318606j, -0.02107352-0.0166436j ,
        -0.0504463 +0.01793493j, -0.00598257+0.00672343j],
       [ 0.0838479 +0.07625424j,  0.07486675-0.01888272j,
         0.01352157+0.0234008j ,  0.06520813+0.01828556j,
         0.

In [13]:
alpha_integral

array([[-7.58989081e-02+3.15076292e-01j, -1.09575212e-01-4.82671289e-03j,
         4.59891632e-02+1.30383357e-01j, -1.21141359e-01+7.62644261e-02j,
         3.08624115e-02+1.91454254e-02j, -5.51427752e-02+1.23301484e-01j,
        -3.85867804e-02-9.90634877e-03j,  1.65003818e-02+9.48043242e-02j,
        -7.90676996e-02+3.34326737e-02j,  2.92015281e-02+2.79933866e-02j],
       [ 1.67584583e-01-1.74262211e-01j,  5.19614033e-02+4.31582481e-02j,
         2.19478607e-02-1.08705305e-01j,  9.81732085e-02+3.57755530e-03j,
        -1.81648899e-02-3.42461281e-02j,  8.35738704e-02-5.31370342e-02j,
         9.47770756e-03+1.71027817e-02j,  2.96467934e-02-7.02601895e-02j,
         5.47482744e-02+1.29892500e-02j, -9.72390082e-03-3.75913642e-02j],
       [-1.99380934e-01+5.53464480e-02j, -8.39059800e-03-4.58557904e-02j,
        -6.86636865e-02+7.39801452e-02j, -6.11708835e-02-4.60996591e-02j,
        -3.00335465e-03+4.25038710e-02j, -8.26210156e-02-2.09872215e-03j,
         6.32234383e-03-8.32669809e-

In [14]:
np.abs(beta_formula - beta_integral)

array([[1.2748903 , 0.85471904, 0.57202864, 0.33194512, 0.2958522 ,
        0.15490581, 0.26880437, 0.23402911, 0.17932303, 0.1560255 ],
       [0.55111694, 0.42978728, 0.2736956 , 0.1471233 , 0.13932578,
        0.06948256, 0.14472118, 0.11611606, 0.08691103, 0.07158174],
       [0.31757477, 0.28511852, 0.16845946, 0.08790749, 0.0902401 ,
        0.04720609, 0.10177477, 0.07249089, 0.05565331, 0.04542003],
       [0.20400774, 0.21159908, 0.11357035, 0.06015356, 0.06829739,
        0.03651403, 0.07922804, 0.0487003 , 0.04018804, 0.03499397],
       [0.1376229 , 0.16686188, 0.07982255, 0.0449514 , 0.05683276,
        0.02916799, 0.0648817 , 0.03365787, 0.031159  , 0.03067833],
       [0.09452077, 0.13666172, 0.05732832, 0.03585263, 0.05005981,
        0.0232353 , 0.05469175, 0.02371278, 0.02531713, 0.02874036],
       [0.06462231, 0.114851  , 0.04182756, 0.03001507, 0.04551072,
        0.0181937 , 0.04694067, 0.01742228, 0.02122267, 0.02759274],
       [0.04299562, 0.0983362 , 0.0312214

In [15]:
np.abs(alpha_formula - alpha_integral)

array([[1.9173167 , 0.7382865 , 0.45641553, 0.47338992, 0.32828516,
        0.4122267 , 0.20788954, 0.18568859, 0.18506908, 0.141269  ],
       [1.078129  , 0.38851255, 0.224128  , 0.2550215 , 0.17034666,
        0.24951085, 0.11825666, 0.1114786 , 0.09112982, 0.06256575],
       [0.7938686 , 0.2807273 , 0.1530237 , 0.17601211, 0.11557218,
        0.19091943, 0.09339831, 0.09315578, 0.05628337, 0.03352502],
       [0.65003264, 0.23212536, 0.12244837, 0.13272864, 0.08625473,
        0.158785  , 0.0831707 , 0.08683961, 0.0378252 , 0.01678245],
       [0.5626196 , 0.20671119, 0.10816235, 0.10431171, 0.06695528,
        0.13704504, 0.07797386, 0.08425734, 0.02835896, 0.00540237],
       [0.50336033, 0.19255786, 0.10171095, 0.08398454, 0.05260054,
        0.12025987, 0.07476212, 0.08286203, 0.02609936, 0.0057196 ],
       [0.46005788, 0.18455286, 0.09925473, 0.06912844, 0.04108538,
        0.10610825, 0.07230556, 0.08165415, 0.02879073, 0.01312748],
       [0.42659485, 0.18013471, 0.0988188